# Day 30 — Deploy the RAG pipeline as an API

Package Week 6's RAG pipeline as an HTTP endpoint: the Lambda handler, the request/response
contract, cold-start mitigation, the API Gateway + IAM infrastructure, auth and rate limiting,
and when Lambda is the wrong choice. The handler runs locally in this notebook exactly as it
would in Lambda; the deployable artifacts get written to this folder.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | The target: a serverless HTTP API | 3 min |
| 1 | The Lambda handler + the contract | 14 min |
| 2 | Cold starts: the enemy of ML on Lambda | 12 min |
| 3 | The infrastructure (SAM template, API Gateway, IAM) | 12 min |
| 4 | Auth, rate limiting, logging, errors | 12 min |
| 5 | When Lambda is the wrong choice | 4 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import json, time, os, hashlib, base64
print("ready")

ready


## 0 — The target (3 min)

```
client --HTTPS--> API Gateway (HTTP API) --> Lambda (RAG handler) --> Bedrock/Anthropic (LLM)
                       |                          |
                   auth, throttle,           retrieve from a
                   request validation        vector store (pgvector / KB)
```

Serverless because RAG traffic is usually **spiky and low-average** — you don't want a GPU (or
even an EC2 box) idling. Lambda scales to zero and bills per millisecond. The catches: a **29s
API Gateway timeout**, cold starts, and no GPU — all manageable for an API-backed RAG service,
all covered below.

## 1 — The Lambda handler + the contract (14 min)

**Module-level init runs once per container** (across many invocations). Put the expensive
setup — clients, the embedded KB, config — at module scope, *not* inside `handler`.

In [2]:
# ============ handler.py (runs locally here; identical in Lambda) ============
# ---- module scope: runs ONCE per cold start, reused by every warm invocation ----
_COLD_START_T = time.time()

def _init():
    # in real life: boto3 clients, load KB embeddings from EFS/S3, read SSM params
    time.sleep(0.15)   # simulate client + small-model init
    return dict(kb={"refund": "Refunds are issued within 5 business days.",
                    "shipping": "Free shipping over $50; 3-5 business days.",
                    "api": "Rate limit is 600 requests/minute; 429 on exceed."},
                model_id=os.environ.get("MODEL_ID", "anthropic.claude-sonnet-4-5"))

STATE = _init()
INIT_MS = (time.time() - _COLD_START_T) * 1000

def _rag_answer(question, state):
    # stand-in for the Day 18 pipeline: retrieve + generate
    q = question.lower()
    hit = next((v for k, v in state["kb"].items() if k in q or any(w in q for w in k.split())), None)
    if not hit:
        return "I don't have information on that.", []
    return hit, [k for k in state["kb"] if k in q]

def _response(status, body, trace_id):
    return {"statusCode": status,
            "headers": {"content-type": "application/json",
                        "x-trace-id": trace_id,
                        "access-control-allow-origin": "*"},
            "body": json.dumps(body)}

def handler(event, context=None):
    trace_id = (getattr(context, "aws_request_id", None)
                or hashlib.md5(json.dumps(event).encode()).hexdigest()[:16])
    t0 = time.time()
    try:
        if event.get("requestContext", {}).get("http", {}).get("method") == "OPTIONS":
            return _response(204, {}, trace_id)                    # CORS preflight
        body = json.loads(event.get("body") or "{}")
        question = (body.get("question") or "").strip()
        if not question:
            return _response(400, {"error": "missing 'question'"}, trace_id)
        if len(question) > 2000:
            return _response(413, {"error": "question too long"}, trace_id)

        answer, sources = _rag_answer(question, STATE)
        latency_ms = round((time.time() - t0) * 1000, 1)
        # structured log line (Day 27) -> goes to CloudWatch as JSON
        print(json.dumps(dict(evt="rag_request", trace_id=trace_id, q_len=len(question),
                              n_sources=len(sources), latency_ms=latency_ms,
                              cold_start_ms=round(INIT_MS, 1))))
        return _response(200, dict(answer=answer, sources=sources,
                                   trace_id=trace_id, latency_ms=latency_ms), trace_id)
    except json.JSONDecodeError:
        return _response(400, {"error": "invalid JSON body"}, trace_id)
    except Exception as e:
        print(json.dumps(dict(evt="error", trace_id=trace_id, error=repr(e))))
        return _response(500, {"error": "internal error", "trace_id": trace_id}, trace_id)
# ============ end handler.py ============
print(f"module init took {INIT_MS:.0f} ms (this is your cold-start floor)")

module init took 154 ms (this is your cold-start floor)


In [3]:
# a local test harness: build API-Gateway-HTTP-API-shaped events and invoke the handler
def api_event(method="POST", path="/ask", body=None, headers=None):
    return {"version": "2.0", "routeKey": f"{method} {path}",
            "rawPath": path, "headers": headers or {"content-type": "application/json"},
            "requestContext": {"http": {"method": method, "path": path}},
            "body": json.dumps(body) if body is not None else None, "isBase64Encoded": False}

class Ctx:  # minimal Lambda context
    aws_request_id = "req-local-abc123"
    def get_remaining_time_in_millis(self): return 29000

for name, ev in [
    ("valid",        api_event(body={"question": "how long do refunds take"})),
    ("missing q",    api_event(body={"foo": "bar"})),
    ("bad json",     {**api_event(), "body": "{not json"}),
    ("out of scope", api_event(body={"question": "what is your stock price"})),
    ("CORS preflight", api_event(method="OPTIONS")),
]:
    r = handler(ev, Ctx())
    print(f"{name:15s} -> {r['statusCode']}  {r['body'][:80]}")

{"evt": "rag_request", "trace_id": "req-local-abc123", "q_len": 24, "n_sources": 1, "latency_ms": 0.0, "cold_start_ms": 153.9}
valid           -> 200  {"answer": "Refunds are issued within 5 business days.", "sources": ["refund"], 
missing q       -> 400  {"error": "missing 'question'"}
bad json        -> 400  {"error": "invalid JSON body"}
{"evt": "rag_request", "trace_id": "req-local-abc123", "q_len": 24, "n_sources": 0, "latency_ms": 0.0, "cold_start_ms": 153.9}
out of scope    -> 200  {"answer": "I don't have information on that.", "sources": [], "trace_id": "req-
CORS preflight  -> 204  {}


### The contract

| | |
| --- | --- |
| **Request** | `POST /ask` · `{"question": "<=2000 chars"}` · header `x-api-key` |
| **200** | `{"answer": str, "sources": [str], "trace_id": str, "latency_ms": float}` |
| **400** | missing/invalid `question` or body |
| **413** | question too long |
| **429** | rate limited (API Gateway usage plan — never reaches Lambda) |
| **500** | `{"error": "internal error", "trace_id": str}` — trace_id lets support find the log |

Every response carries `x-trace-id`; every log line carries `trace_id` (Day 27). "It gave me a
bad answer, here's the trace id" → one CloudWatch Logs Insights query.

## 2 — Cold starts (12 min)

A **cold start** = Lambda spins up a fresh container, runs your module-level code, *then*
handles the request. For plain Python it's ~100–300ms. For ML it can be **seconds** — loading
an embedding model, big deps (`torch`, `transformers`), reading a KB.

| Cause | Fix |
| ----- | --- |
| Big deps (`torch`, `sentence-transformers`) | don't. Use an **API embedding** (Bedrock Titan / Cohere) or a Bedrock **Knowledge Base** so the Lambda carries no model |
| Loading a local model at init | package it in a **container image** (up to 10 GB) or mount from **EFS**; still slow — prefer API embeddings |
| Reading the KB / index at init | keep it in the vector DB (pgvector/KB), not in the Lambda; the handler just queries |
| Big zip / slow import | trim deps; lazy-import anything not on the hot path; `--platform manylinux2014` wheels |
| First request after idle | **provisioned concurrency** (keep N containers warm) or a scheduled "ping" |

In [4]:
# measure the difference: model-in-lambda vs API-embedding-in-lambda
def init_with_local_model():
    time.sleep(2.4)   # load sentence-transformers + torch + weights
    return "local model ready"

def init_with_api_embeddings():
    time.sleep(0.12)  # just a boto3 client; embeddings happen over the network per request
    return "boto3 client ready"

for name, fn in [("local model in Lambda", init_with_local_model),
                 ("API embeddings (Bedrock/Titan)", init_with_api_embeddings)]:
    t = time.time(); fn(); print(f"{name:32s} cold start ~{(time.time()-t)*1000:.0f} ms")

print("\ncost of cold starts: at 1 req/s with 5% cold, a 2.4s init adds ~120ms to p50 and")
print("seconds to p99. Provisioned concurrency removes it -- for a fixed hourly fee per unit.")

local model in Lambda            cold start ~2405 ms
API embeddings (Bedrock/Titan)   cold start ~125 ms

cost of cold starts: at 1 req/s with 5% cold, a 2.4s init adds ~120ms to p50 and
seconds to p99. Provisioned concurrency removes it -- for a fixed hourly fee per unit.


**The design rule for RAG on Lambda: keep no model in the Lambda.** Embeddings via a Bedrock
API call (or a Knowledge Base), generation via Bedrock/Anthropic, retrieval via a query to
pgvector. The Lambda is then a thin ~50 MB function that cold-starts in ~200 ms.

## 3 — The infrastructure (12 min)

`template.yaml` (AWS SAM) — API Gateway HTTP API + Lambda + IAM, deployed with
`sam build && sam deploy --guided`. Written to this folder.

In [5]:
TEMPLATE = '''
AWSTemplateFormatVersion: "2010-09-09"
Transform: AWS::Serverless-2016-10-31
Description: RAG API (Day 30)

Globals:
  Function:
    Timeout: 25            # < the 29s API Gateway hard limit
    MemorySize: 1024       # more memory = more CPU = faster; tune with Lambda Power Tuning
    Runtime: python3.12
    Architectures: [arm64] # cheaper + faster for this workload
    Environment:
      Variables:
        MODEL_ID: anthropic.claude-sonnet-4-5-20250929-v1:0
        VECTOR_DB_SECRET: !Ref VectorDbSecretArn
    Tracing: Active        # X-Ray

Parameters:
  VectorDbSecretArn: { Type: String }

Resources:
  RagApi:
    Type: AWS::Serverless::HttpApi
    Properties:
      Auth:
        ApiKeyRequired: true          # or a Lambda authorizer / Cognito
      Throttle: { RateLimit: 50, BurstLimit: 100 }
      CorsConfiguration:
        AllowOrigins: ["https://app.example.com"]
        AllowMethods: [POST, OPTIONS]
        AllowHeaders: [content-type, x-api-key]

  RagFunction:
    Type: AWS::Serverless::Function
    Properties:
      Handler: handler.handler
      CodeUri: ./src
      ReservedConcurrentExecutions: 100        # cap blast radius / cost
      ProvisionedConcurrencyConfig:
        ProvisionedConcurrentExecutions: 2     # keep 2 warm (remove cold starts on the hot path)
      Policies:
        - Statement:
            - Effect: Allow
              Action: ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"]
              Resource: "arn:aws:bedrock:*::foundation-model/anthropic.*"
            - Effect: Allow
              Action: ["secretsmanager:GetSecretValue"]
              Resource: !Ref VectorDbSecretArn
      Events:
        Ask:
          Type: HttpApi
          Properties: { ApiId: !Ref RagApi, Method: POST, Path: /ask }

Outputs:
  ApiUrl: { Value: !Sub "https://${RagApi}.execute-api.${AWS::Region}.amazonaws.com" }
'''
import pathlib
pathlib.Path("template.yaml").write_text(TEMPLATE.strip() + "\n")
pathlib.Path("src").mkdir(exist_ok=True)
print("wrote template.yaml + src/  (put handler.py in src/, add requirements.txt)")
print(TEMPLATE[:900])

wrote template.yaml + src/  (put handler.py in src/, add requirements.txt)

AWSTemplateFormatVersion: "2010-09-09"
Transform: AWS::Serverless-2016-10-31
Description: RAG API (Day 30)

Globals:
  Function:
    Timeout: 25            # < the 29s API Gateway hard limit
    MemorySize: 1024       # more memory = more CPU = faster; tune with Lambda Power Tuning
    Runtime: python3.12
    Architectures: [arm64] # cheaper + faster for this workload
    Environment:
      Variables:
        MODEL_ID: anthropic.claude-sonnet-4-5-20250929-v1:0
        VECTOR_DB_SECRET: !Ref VectorDbSecretArn
    Tracing: Active        # X-Ray

Parameters:
  VectorDbSecretArn: { Type: String }

Resources:
  RagApi:
    Type: AWS::Serverless::HttpApi
    Properties:
      Auth:
        ApiKeyRequired: true          # or a Lambda authorizer / Cognito
      Throttle: { RateLimit: 50, BurstLimit: 100 }
      CorsConfiguration:
        AllowOrigins: ["https://app.example.com"]
        AllowMe


Key choices in the template:

- **Timeout 25s** — must be under API Gateway's 29s hard cap; if your RAG call can exceed
  that, you need streaming (§4) or a different compute (§5).
- **arm64** — ~20% cheaper and often faster for this workload.
- **ReservedConcurrentExecutions** — caps how many Lambdas can run at once: bounds cost and
  protects your downstream (Bedrock quotas, the DB connection pool).
- **ProvisionedConcurrency 2** — the hot path is always warm; bursts beyond 2 pay a cold start.
- **Least-privilege IAM** — only `bedrock:InvokeModel` on Anthropic models and read on the one
  secret. No `*`.
- Deploy: `sam build && sam deploy --guided`. Roll back: `sam deploy` deploys a new version;
  use `AutoPublishAlias` + `DeploymentPreference: Canary10Percent5Minutes` for gradual rollout.

## 4 — Auth, rate limiting, logging, errors (12 min)

- **Auth.** Simplest: API Gateway **API keys + usage plans** (`ApiKeyRequired: true`) — per-key
  rate limits and quotas, enforced *before* Lambda runs (so a throttled caller costs you
  nothing). For user-facing: a **Lambda authorizer** or **Cognito** JWT.
- **Rate limiting.** Usage-plan `RateLimit`/`BurstLimit` (steady + burst), plus per-account
  Lambda `ReservedConcurrentExecutions`. A 429 from API Gateway never reaches your code.
- **Request validation.** API Gateway can reject malformed bodies at the edge (JSON schema on
  the route) — cheaper than validating in Lambda, though you still guard in code.
- **Logging.** One structured JSON line per request to CloudWatch (Day 27): `trace_id`,
  `q_len`, `latency_ms`, `n_sources`, `cold_start_ms`, downstream `usage`/`cost`. Query with
  Logs Insights. Turn on **X-Ray** for cross-service traces (API GW → Lambda → Bedrock).
- **Errors.** Map exceptions to HTTP status (400 client, 429 downstream throttle, 500 else),
  always return the `trace_id`, never leak stack traces to the client.
- **Secrets.** DB creds / API keys from Secrets Manager or SSM Parameter Store, fetched at
  init and cached — never in env vars in plaintext, never in the code.

In [6]:
# a tiny CloudWatch Logs Insights query you'd actually run
INSIGHTS_QUERY = '''
fields @timestamp, trace_id, latency_ms, n_sources, cold_start_ms
| filter evt = "rag_request"
| stats
    count(*) as requests,
    avg(latency_ms) as avg_ms,
    pct(latency_ms, 95) as p95_ms,
    avg(cold_start_ms) as avg_cold_ms,
    sum(n_sources = 0) as zero_source_answers
  by bin(5m)
'''
print(INSIGHTS_QUERY)
print("-> a spike in zero_source_answers = retrieval is broken. p95_ms climbing = downstream")
print("   slow or cold starts. This dashboard is your production monitoring (Day 25).")


fields @timestamp, trace_id, latency_ms, n_sources, cold_start_ms
| filter evt = "rag_request"
| stats
    count(*) as requests,
    avg(latency_ms) as avg_ms,
    pct(latency_ms, 95) as p95_ms,
    avg(cold_start_ms) as avg_cold_ms,
    sum(n_sources = 0) as zero_source_answers
  by bin(5m)

-> a spike in zero_source_answers = retrieval is broken. p95_ms climbing = downstream
   slow or cold starts. This dashboard is your production monitoring (Day 25).


In [7]:
# streaming from Lambda (for answers that would blow the 29s API Gateway limit, or for UX):
# use a Lambda FUNCTION URL with RESPONSE_STREAM mode (not API Gateway, which buffers).
STREAMING_HANDLER = '''
import awslambdaric  # runtime provides streamifyResponse

@awslambdaric.streamify_response          # (JS: awslambda.streamifyResponse)
def handler(event, response_stream, context):
    question = json.loads(event["body"])["question"]
    chunks = retrieve(question)
    with bedrock.converse_stream(modelId=MODEL_ID, messages=build(question, chunks)) as s:
        for ev in s:
            if "contentBlockDelta" in ev:
                response_stream.write(ev["contentBlockDelta"]["delta"]["text"].encode())
    response_stream.end()
'''
print(STREAMING_HANDLER)
print("Function URLs support response streaming up to 15 min / 20 MB; API Gateway does not.")


import awslambdaric  # runtime provides streamifyResponse

@awslambdaric.streamify_response          # (JS: awslambda.streamifyResponse)
def handler(event, response_stream, context):
    question = json.loads(event["body"])["question"]
    chunks = retrieve(question)
    with bedrock.converse_stream(modelId=MODEL_ID, messages=build(question, chunks)) as s:
        for ev in s:
            if "contentBlockDelta" in ev:
                response_stream.write(ev["contentBlockDelta"]["delta"]["text"].encode())
    response_stream.end()

Function URLs support response streaming up to 15 min / 20 MB; API Gateway does not.


## 5 — When Lambda is the wrong choice (4 min)

| Situation | Better fit |
| --------- | ---------- |
| Requests routinely > 29s (long agent loops, huge generations) | **ECS/Fargate** or **App Runner** behind an ALB; or Lambda **Function URL** streaming (15 min) |
| Steady high RPS (cold starts + per-ms billing add up; you'd keep 50 warm anyway) | **Fargate / App Runner / EKS** — a always-on container is cheaper past a break-even |
| Need a GPU (self-hosted embeddings or generation) | **ECS on GPU / EKS / SageMaker endpoint / Bedrock** — Lambda has no GPU |
| Big model in-process, multi-GB, slow to load | container on ECS with a warm pool; or move the model behind an API |
| WebSocket / long-lived connections | API Gateway WebSocket + Lambda, or Fargate |
| Very latency-sensitive, p99 must be tight | provisioned-concurrency Lambda, or Fargate with a warm pool |

**Lambda is right for RAG when:** traffic is spiky/low-average, each request finishes in a few
seconds, and the Lambda carries no model (embeddings + generation via API). That's the common
case, and it's why this is the default first deployment.

## 6 — Exercises

1. **Add request-schema validation** at the handler: reject bodies with unexpected keys, a
   non-string `question`, or `top_k` outside 1–10. Return 422 with a field-level error list.
   Test with 5 malformed events.
2. **Idempotency.** Accept an `Idempotency-Key` header; cache `(key -> response)` for 10
   minutes (a module-level dict, or DynamoDB in real life) and return the cached response on a
   repeat. Show a retried request returns the identical body + a `x-idempotent-replay: true`
   header.
3. **Timeout guard.** Use `context.get_remaining_time_in_millis()` to abort the downstream
   call and return a 503 with `Retry-After` if fewer than 3s remain, rather than being
   killed mid-request. Simulate with a slow `_rag_answer`.
4. **Power tuning.** For `MemorySize` in `[512, 1024, 1769, 3008]`, model latency as
   `init/mem_factor + fixed` and cost as `mem_gb * duration_s * price`. Find the
   cost-minimising and the latency-minimising memory. (Real tool: AWS Lambda Power Tuning.)
5. **Canary rollout.** Sketch the `DeploymentPreference` + a CloudWatch alarm on the function's
   error rate that auto-rolls-back a bad deploy. What metric and threshold?
6. **Cost model.** At 30 req/s average, 400ms mean duration, 1024 MB, arm64: compute the
   monthly Lambda + API Gateway cost. Compare to a single always-on Fargate task
   (0.5 vCPU / 1 GB) that can handle the same load. Where's the break-even RPS?

> **Attempt every exercise and the quiz first.** The worked solutions and the answer key live in [`solutions/solutions.ipynb`](solutions/solutions.ipynb) — open it only to check your work, not to start.

## Self-check quiz


1. Why put client/model init at module scope instead of inside `handler`?
2. What's the API Gateway timeout, and what are your options if a RAG call can exceed it?
3. Name the single most important design rule for RAG on Lambda re: cold starts.
4. How does API-key + usage-plan auth save you money vs checking auth in the Lambda?
5. What does `ReservedConcurrentExecutions` protect?
6. Give two situations where you should not use Lambda for this, and what to use instead.
7. A user reports a wrong answer. What do you ask them for, and what do you do with it?

## Course complete

Weeks 2–10, from context windows to a deployed, observable, evaluated RAG API. Every concept
built from scratch first, then connected to the real tool. Keep the `NOTES.md` files; re-run a
`lesson.ipynb` when a concept goes fuzzy.

**Good next concepts:** fine-tuning for tool-use / function-calling reliability; multi-agent
orchestration patterns (supervisor/worker, debate); prompt-injection & jailbreak defense in
depth; RAG evaluation with RAGAS/DeepEval on a real corpus; and cost/latency profiling of your
own production traffic with the `claude-api` skill's `cost-optimize` flow.